# Lab 19. Neural Networks for Time Series with TensorFlow/Keras

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-19-neural-networks-for-time-series-lab.ipynb)

This lab is designed for independent study. It explains the mathematical and modeling ideas before the programming steps.

In this lab, you will learn how to turn a time series into a supervised learning data set, train a neural network forecaster using TensorFlow/Keras, compare it with simple baselines, and diagnose whether the model is learning something meaningful.

## Learning goals

By the end of this lab, you should be able to:

1. Convert a univariate time series into input-label windows.
2. Explain why chronological splitting is required for time series.
3. Build a simple neural network forecaster using TensorFlow/Keras.
4. Compare neural network forecasts with naive and seasonal naive baselines.
5. Use validation loss and residual plots to detect overfitting or underfitting.
6. Explain how scaling can introduce data leakage if done incorrectly.
7. Use an AI assistant to critique a neural forecasting workflow without blindly trusting it.

## Connection to the chapter

Chapter 19 introduced neural networks for time series. The key idea is simple but powerful: instead of specifying a linear ARMA form, we let a neural network learn a nonlinear function from a recent window of observations to future values.

## 1. Background: from a time series to supervised learning

Suppose we observe a scalar time series

$$
x_0, x_1, x_2, \ldots, x_{T-1}.
$$

A neural network does not directly understand "time series". It understands arrays. Therefore, we construct examples using sliding windows.

Choose:

- input length $L$,
- forecast horizon $H$.

For each possible starting time $i$, define the input vector

$$
X_i = (x_i, x_{i+1}, \ldots, x_{i+L-1})
$$

and the label vector

$$
y_i = (x_{i+L}, x_{i+L+1}, \ldots, x_{i+L+H-1}).
$$

The training data set is then a collection of pairs

$$
D = \{(X_i, y_i)\}.
$$

A neural network learns a function

$$
f_\theta: R^L \to R^H
$$

so that

$$
f_\theta(X_i) \approx y_i.
$$

The parameter vector $\theta$ contains the weights and biases of the network. We choose $\theta$ by minimizing a loss function such as mean squared error.

### Important warning: chronological splitting

For time series, we should not randomly split the windows into training and testing sets. Random splitting lets the model see examples from the future during training. This is a form of data leakage.

Instead, use chronological splitting:

1. train on the earliest windows,
2. validate on the next block of time,
3. test on the final block of time.

## 2. Setup

This lab uses TensorFlow/Keras. Google Colab usually has TensorFlow installed. If you run this notebook locally and TensorFlow is missing, install it first using

```bash
pip install tensorflow
```

Then restart the kernel.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import tensorflow as tf
except ModuleNotFoundError as err:
    raise ModuleNotFoundError(
        "TensorFlow is not installed. In Google Colab it is usually available. "
        "For local use, run: pip install tensorflow, then restart the kernel."
    ) from err

from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(7339)
tf.keras.utils.set_random_seed(7339)

# Keep TensorFlow small and stable for a teaching notebook.
try:
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
except Exception:
    pass

print("TensorFlow version:", tf.__version__)

## 3. Simulate a time series with trend, seasonality, and nonlinear dependence

Real time series often contain multiple components:

$$
x_t = \text{trend}_t + \text{seasonality}_t + \text{dependence}_t + \text{noise}_t.
$$

Here we simulate a series that has:

- a slow trend,
- a daily-like cycle with period 24,
- a longer cycle with period 168,
- nonlinear dependence on previous values,
- random noise.

This gives the neural network something meaningful to learn.

In [ ]:
def simulate_series(n=1200, seed=7339):
    rng = np.random.default_rng(seed)
    t = np.arange(n)

    trend = 0.0025 * t
    daily = 1.2 * np.sin(2 * np.pi * t / 24)
    weekly = 0.7 * np.sin(2 * np.pi * t / 168)
    noise = rng.normal(0, 0.35, size=n)

    x = np.zeros(n)
    for i in range(2, n):
        nonlinear_term = 0.18 * np.tanh(x[i - 1])
        x[i] = (
            0.55 * x[i - 1]
            - 0.18 * x[i - 2]
            + trend[i]
            + daily[i]
            + weekly[i]
            + nonlinear_term
            + noise[i]
        )
    return t, x

t, x = simulate_series()

series = pd.DataFrame({"t": t, "x": x})
series.head()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(series["t"], series["x"])
plt.title("Simulated time series")
plt.xlabel("time")
plt.ylabel("x")
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(series["t"].iloc[:240], series["x"].iloc[:240])
plt.title("First 240 observations")
plt.xlabel("time")
plt.ylabel("x")
plt.show()

### Checkpoint 1

Before training any neural network, answer these questions:

1. Does the series look stationary?
2. What patterns appear visually predictable?
3. Why might a nonlinear model help here?
4. What simple baseline forecasts should a neural network beat?

## 4. Create input-label windows

We will use the previous $L=48$ observations to predict the next $H=12$ observations.

That means the model sees two days of hourly-like history and predicts the next half day.

In [ ]:
def make_windows(values, input_length, horizon):
    # Return arrays X and y from a one-dimensional time series.
    values = np.asarray(values, dtype=np.float32)
    X, y = [], []
    last_start = len(values) - input_length - horizon + 1
    for start in range(last_start):
        end_input = start + input_length
        end_label = end_input + horizon
        X.append(values[start:end_input])
        y.append(values[end_input:end_label])
    return np.array(X), np.array(y)

L = 48
H = 12

X, y = make_windows(series["x"].values, input_length=L, horizon=H)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("One input window has length", X.shape[1])
print("One label window has length", y.shape[1])

In [ ]:
example_id = 200
input_times = np.arange(example_id, example_id + L)
label_times = np.arange(example_id + L, example_id + L + H)

plt.figure(figsize=(10, 4))
plt.plot(input_times, X[example_id], marker="o", label="input window")
plt.plot(label_times, y[example_id], marker="x", label="label window")
plt.axvline(example_id + L - 0.5, linestyle="--", label="forecast origin")
plt.title("One supervised learning example")
plt.xlabel("time")
plt.ylabel("x")
plt.legend()
plt.show()

## 5. Chronological train-validation-test split

We split the windowed examples in chronological order.

The model should be trained on the past and evaluated on the future.

In [ ]:
n_samples = len(X)
n_train = int(0.70 * n_samples)
n_val = int(0.15 * n_samples)

X_train_raw = X[:n_train]
y_train_raw = y[:n_train]

X_val_raw = X[n_train:n_train + n_val]
y_val_raw = y[n_train:n_train + n_val]

X_test_raw = X[n_train + n_val:]
y_test_raw = y[n_train + n_val:]

print("Training samples:", X_train_raw.shape[0])
print("Validation samples:", X_val_raw.shape[0])
print("Test samples:", X_test_raw.shape[0])

## 6. Scaling without leakage

Neural networks usually train better when inputs are scaled.

Important rule: compute scaling constants using the training set only.

If you compute the mean and standard deviation using the full data set, the model indirectly sees information from the validation and test periods.

In [ ]:
train_mean = X_train_raw.mean()
train_std = X_train_raw.std()

X_train = (X_train_raw - train_mean) / train_std
X_val = (X_val_raw - train_mean) / train_std
X_test = (X_test_raw - train_mean) / train_std

y_train = (y_train_raw - train_mean) / train_std
y_val = (y_val_raw - train_mean) / train_std
y_test = (y_test_raw - train_mean) / train_std

print("Training mean used for scaling:", train_mean)
print("Training standard deviation used for scaling:", train_std)

## 7. Baseline forecasts

A neural network is useful only if it improves on simple baselines.

We will compare with:

1. **Last-value baseline:** repeat the most recent observation.
2. **Seasonal naive baseline:** repeat the value from one seasonal period ago.

For an input window $X_i$, the last-value baseline predicts

$$
\hat y_{i,h} = x_{i+L-1}.
$$

If the seasonal period is 24 and the horizon is no more than 24, the seasonal naive baseline predicts

$$
\hat y_{i,h} = x_{i+L-24+h}.
$$

In [ ]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def last_value_baseline(X_raw, horizon):
    last = X_raw[:, -1]
    return np.repeat(last[:, None], horizon, axis=1)

def seasonal_naive_baseline(X_raw, horizon, period=24):
    if X_raw.shape[1] < period:
        raise ValueError("Input length must be at least one seasonal period.")
    return X_raw[:, -period:-period + horizon]

last_pred = last_value_baseline(X_test_raw, H)
seasonal_pred = seasonal_naive_baseline(X_test_raw, H, period=24)

baseline_results = pd.DataFrame({
    "model": ["last value", "seasonal naive"],
    "MAE": [mae(y_test_raw, last_pred), mae(y_test_raw, seasonal_pred)],
    "RMSE": [rmse(y_test_raw, last_pred), rmse(y_test_raw, seasonal_pred)]
})

baseline_results

### Checkpoint 2

A common mistake is to train a sophisticated neural network without checking baselines.

Before continuing, write down:

1. Which baseline is stronger here?
2. Why might the seasonal naive method be strong?
3. What would it mean if the neural network is worse than both baselines?

## 8. Build a TensorFlow/Keras MLP forecaster

Our first neural network is a multilayer perceptron. It treats the input window as a vector.

The model is

$$
X_i \mapsto f_\theta(X_i) \in R^H.
$$

The output layer has $H$ units because we predict all $H$ future values at once. This is called a **direct multi-output forecast**.

In [ ]:
def build_mlp(input_length, horizon, width=64, dropout=0.10, learning_rate=0.001):
    model = keras.Sequential([
        layers.Input(shape=(input_length,)),
        layers.Dense(width, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(width, activation="relu"),
        layers.Dense(horizon)
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

mlp = build_mlp(L, H, width=64, dropout=0.10)
mlp.summary()

## 9. Train the model

We use early stopping. Early stopping monitors validation loss and stops training when validation performance no longer improves.

This is a practical way to reduce overfitting.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history = mlp.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

history_df = pd.DataFrame(history.history)
history_df.tail()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_df["loss"], label="training loss")
plt.plot(history_df["val_loss"], label="validation loss")
plt.title("Training history")
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.legend()
plt.show()

### Checkpoint 3

Use the training-history plot to answer:

1. Is the model underfitting, overfitting, or about right?
2. Did validation loss stop improving before training loss stopped improving?
3. Would you increase model size, decrease model size, or change regularization?

## 10. Evaluate the neural network on the test set

The model predicts standardized values. We convert predictions back to the original scale before computing final metrics.

In [ ]:
y_pred_scaled = mlp.predict(X_test, verbose=0)
y_pred = y_pred_scaled * train_std + train_mean

nn_results = pd.DataFrame({
    "model": ["TensorFlow MLP"],
    "MAE": [mae(y_test_raw, y_pred)],
    "RMSE": [rmse(y_test_raw, y_pred)]
})

all_results = pd.concat([baseline_results, nn_results], ignore_index=True)
all_results

In [ ]:
def plot_forecast_example(example_index):
    input_window = X_test_raw[example_index]
    true_future = y_test_raw[example_index]
    pred_future = y_pred[example_index]
    seasonal_future = seasonal_pred[example_index]

    input_axis = np.arange(L)
    future_axis = np.arange(L, L + H)

    plt.figure(figsize=(10, 4))
    plt.plot(input_axis, input_window, marker="o", label="input")
    plt.plot(future_axis, true_future, marker="x", label="true future")
    plt.plot(future_axis, pred_future, marker="o", label="TensorFlow MLP")
    plt.plot(future_axis, seasonal_future, marker="s", label="seasonal naive")
    plt.axvline(L - 0.5, linestyle="--", label="forecast origin")
    plt.title(f"Forecast example {example_index}")
    plt.xlabel("relative time")
    plt.ylabel("x")
    plt.legend()
    plt.show()

plot_forecast_example(0)
plot_forecast_example(25)
plot_forecast_example(75)

## 11. Horizon-wise errors

A model may do well for the first few forecast steps but poorly for longer horizons.

We compute error separately for each horizon $h=1,\ldots,H$.

In [ ]:
horizon_mae_nn = np.mean(np.abs(y_test_raw - y_pred), axis=0)
horizon_mae_last = np.mean(np.abs(y_test_raw - last_pred), axis=0)
horizon_mae_seasonal = np.mean(np.abs(y_test_raw - seasonal_pred), axis=0)

horizons = np.arange(1, H + 1)

plt.figure(figsize=(8, 4))
plt.plot(horizons, horizon_mae_nn, marker="o", label="TensorFlow MLP")
plt.plot(horizons, horizon_mae_last, marker="o", label="last value")
plt.plot(horizons, horizon_mae_seasonal, marker="o", label="seasonal naive")
plt.title("Horizon-wise MAE")
plt.xlabel("forecast horizon")
plt.ylabel("MAE")
plt.legend()
plt.show()

pd.DataFrame({
    "horizon": horizons,
    "MLP_MAE": horizon_mae_nn,
    "last_value_MAE": horizon_mae_last,
    "seasonal_naive_MAE": horizon_mae_seasonal
})

## 12. Residual diagnostics

Define the forecast residuals by

$$
e_{i,h} = y_{i,h} - \hat y_{i,h}.
$$

For a good forecast model, residuals should not show obvious systematic patterns. They do not need to be independent in every strict sense, but persistent structure suggests that the model is missing information.

In [ ]:
residuals = y_test_raw - y_pred
one_step_resid = residuals[:, 0]

plt.figure(figsize=(10, 4))
plt.plot(one_step_resid)
plt.axhline(0, linestyle="--")
plt.title("One-step-ahead residuals on the test set")
plt.xlabel("test window index")
plt.ylabel("residual")
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(one_step_resid, bins=30, edgecolor="black")
plt.title("Histogram of one-step-ahead residuals")
plt.xlabel("residual")
plt.ylabel("count")
plt.show()

In [ ]:
def sample_acf(values, max_lag):
    values = np.asarray(values)
    centered = values - values.mean()
    denom = np.sum(centered ** 2)
    acfs = []
    for lag in range(max_lag + 1):
        if lag == 0:
            acfs.append(1.0)
        else:
            acfs.append(np.sum(centered[:-lag] * centered[lag:]) / denom)
    return np.array(acfs)

max_lag = 30
acf_resid = sample_acf(one_step_resid, max_lag)

plt.figure(figsize=(8, 4))
plt.stem(np.arange(max_lag + 1), acf_resid)
plt.axhline(0, color="black", linewidth=1)
plt.title("Sample ACF of one-step residuals")
plt.xlabel("lag")
plt.ylabel("ACF")
plt.show()

### Checkpoint 4

Look at the residual plots.

1. Are residuals centered around zero?
2. Is there changing variance?
3. Is there visible autocorrelation?
4. What change to the model might reduce the residual structure?

## 13. A multivariate neural forecasting example

The previous model used only past values of the series. We can often help a neural network by giving it time features.

For a periodic effect with period $P$, use Fourier features:

$$
\sin(2\pi t/P), \qquad \cos(2\pi t/P).
$$

These features allow the model to know where it is in the daily or weekly cycle.

In [ ]:
def make_multivariate_windows(values, times, input_length, horizon):
    values = np.asarray(values, dtype=np.float32)
    times = np.asarray(times, dtype=np.float32)

    # Features available at input times.
    feature_matrix = np.column_stack([
        values,
        np.sin(2 * np.pi * times / 24),
        np.cos(2 * np.pi * times / 24),
        np.sin(2 * np.pi * times / 168),
        np.cos(2 * np.pi * times / 168),
    ]).astype(np.float32)

    X_multi, y_multi = [], []
    last_start = len(values) - input_length - horizon + 1
    for start in range(last_start):
        end_input = start + input_length
        end_label = end_input + horizon
        X_multi.append(feature_matrix[start:end_input, :])
        y_multi.append(values[end_input:end_label])
    return np.array(X_multi), np.array(y_multi)

X_multi, y_multi = make_multivariate_windows(series["x"].values, series["t"].values, L, H)

X_multi_train_raw = X_multi[:n_train]
X_multi_val_raw = X_multi[n_train:n_train + n_val]
X_multi_test_raw = X_multi[n_train + n_val:]

# Scale only the value channel using training data. The sine/cosine features are already scaled.
value_mean = X_multi_train_raw[:, :, 0].mean()
value_std = X_multi_train_raw[:, :, 0].std()

def scale_multivariate(X_raw):
    X_scaled = X_raw.copy()
    X_scaled[:, :, 0] = (X_scaled[:, :, 0] - value_mean) / value_std
    return X_scaled

X_multi_train = scale_multivariate(X_multi_train_raw)
X_multi_val = scale_multivariate(X_multi_val_raw)
X_multi_test = scale_multivariate(X_multi_test_raw)

y_multi_train = (y_train_raw - value_mean) / value_std
y_multi_val = (y_val_raw - value_mean) / value_std
y_multi_test = (y_test_raw - value_mean) / value_std

print("Multivariate input shape:", X_multi_train.shape)

In [ ]:
def build_multivariate_mlp(input_length, n_features, horizon, width=64):
    model = keras.Sequential([
        layers.Input(shape=(input_length, n_features)),
        layers.Flatten(),
        layers.Dense(width, activation="relu"),
        layers.Dropout(0.10),
        layers.Dense(width, activation="relu"),
        layers.Dense(horizon)
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

multi_mlp = build_multivariate_mlp(L, X_multi_train.shape[-1], H, width=64)

history_multi = multi_mlp.fit(
    X_multi_train, y_multi_train,
    validation_data=(X_multi_val, y_multi_val),
    epochs=80,
    batch_size=32,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)],
    verbose=0
)

multi_pred_scaled = multi_mlp.predict(X_multi_test, verbose=0)
multi_pred = multi_pred_scaled * value_std + value_mean

multi_results = pd.DataFrame({
    "model": ["TensorFlow MLP with Fourier features"],
    "MAE": [mae(y_test_raw, multi_pred)],
    "RMSE": [rmse(y_test_raw, multi_pred)]
})

pd.concat([all_results, multi_results], ignore_index=True)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_df["val_loss"].values, label="univariate validation loss")
plt.plot(pd.DataFrame(history_multi.history)["val_loss"].values, label="multivariate validation loss")
plt.title("Validation loss comparison")
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.legend()
plt.show()

## 14. Capacity experiment

A larger network can fit more complicated functions, but it can also overfit.

We compare three network widths. Keep the experiment small so it runs quickly in Colab.

In [ ]:
widths = [16, 64, 128]
capacity_rows = []

for width in widths:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(7339 + width)

    model = build_mlp(L, H, width=width, dropout=0.10)
    hist = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=60,
        batch_size=32,
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)],
        verbose=0
    )
    pred_scaled = model.predict(X_test, verbose=0)
    pred = pred_scaled * train_std + train_mean
    capacity_rows.append({
        "width": width,
        "epochs_run": len(hist.history["loss"]),
        "best_val_loss": float(np.min(hist.history["val_loss"])),
        "test_MAE": mae(y_test_raw, pred),
        "test_RMSE": rmse(y_test_raw, pred),
    })

capacity_results = pd.DataFrame(capacity_rows)
capacity_results

### Checkpoint 5

Interpret the capacity experiment.

1. Did the widest model give the best test error?
2. Did the best validation loss correspond to the best test error?
3. What signs of overfitting or underfitting do you see?

## 15. AI-assisted study prompts

Use an AI assistant as a critic, not as an oracle. Copy the prompt below and fill in your results.

### Prompt 1: leakage audit

I built a TensorFlow neural network forecaster using sliding windows. The input length is 48 and the horizon is 12. I split the windowed data chronologically into training, validation, and test sets. I computed scaling constants using only the training windows. Please audit this workflow for possible time-series leakage. Give a checklist of things I should verify manually.

### Prompt 2: residual interpretation

Here are my test-set metrics and a description of my residual plots: [insert metrics and observations]. Please suggest three possible reasons why residuals still have autocorrelation. For each reason, suggest one diagnostic plot or experiment.

### Prompt 3: model comparison

My seasonal naive baseline has MAE [insert number], my univariate MLP has MAE [insert number], and my MLP with Fourier features has MAE [insert number]. Please help me write a short model-comparison paragraph for a technical report. Do not overclaim causality.

## 16. Exercises

### Exercise 1: change the input length

Repeat the experiment with $L=24$, $L=72$, and $L=168$. Which input length gives the best validation loss? Which gives the best test MAE?

### Exercise 2: change the forecast horizon

Try $H=1$, $H=6$, and $H=24$. How does the horizon affect forecasting difficulty?

### Exercise 3: compare direct and recursive forecasting

In this lab we predicted all $H$ future values directly. Build a one-step model and use it recursively to predict $H$ steps. Compare the errors.

### Exercise 4: regularization

Try dropout values 0, 0.10, 0.30, and 0.50. What happens to training loss, validation loss, and test error?

### Exercise 5: real-data extension

Choose a real time series. Apply the same workflow:

1. time plot,
2. baseline forecasts,
3. chronological split,
4. scaling from training data only,
5. TensorFlow model,
6. test-set comparison,
7. residual diagnostics.

## 17. Mini-project

Write a short report with the following sections:

1. **Data description.** What does the time series represent?
2. **Forecasting task.** What is the input length and forecast horizon?
3. **Baselines.** Which baselines did you use?
4. **Neural network model.** Describe the architecture and training procedure.
5. **Results.** Report MAE and RMSE.
6. **Diagnostics.** Include residual plots and horizon-wise errors.
7. **Conclusion.** State whether the neural network is justified compared with the baselines.

A good report should not simply say "the neural network is modern." It should explain whether the neural network adds predictive value.

## 18. Final checklist

Before leaving this lab, make sure you can answer each question.

- What is a sliding window?
- Why do we split time series chronologically?
- Why is scaling using the full data set a leakage problem?
- What is a direct multi-output forecast?
- Why must a neural network be compared with naive baselines?
- What does early stopping do?
- How can residual plots reveal a missing pattern?
- Why can Fourier features help a neural time-series model?